In [2]:
import pandas as pd
import numpy as np
from decimal import Decimal, ROUND_HALF_UP

In [3]:
case_df = pd.read_csv('time_series_covid19_confirmed_US.csv')
case_df.head()

,UID,iso2,iso3,code3,FIPS,Admin2,Province_State,Country_Region,Lat,Long_,...,2/28/23,3/1/23,3/2/23,3/3/23,3/4/23,3/5/23,3/6/23,3/7/23,3/8/23,3/9/23
0,84001001,US,USA,840,1001.0,Autauga,Alabama,US,32.539527,-86.644082,...,19732,19759,19759,19759,19759,19759,19759,19759,19790,19790
1,84001003,US,USA,840,1003.0,Baldwin,Alabama,US,30.727750,-87.722071,...,69641,69767,69767,69767,69767,69767,69767,69767,69860,69860
2,84001005,US,USA,840,1005.0,Barbour,Alabama,US,31.868263,-85.387129,...,7451,7474,7474,7474,7474,7474,7474,7474,7485,7485
3,84001007,US,USA,840,1007.0,Bibb,Alabama,US,32.996421,-87.125115,...,8067,8087,8087,8087,8087,8087,8087,8087,8091,8091
4,84001009,US,USA,840,1009.0,Blount,Alabama,US,33.982109,-86.567906,...,18616,18673,18673,18673,18673,18673,18673,18673,18704,18704


In [4]:
death_df = pd.read_csv('time_series_covid19_deaths_US.csv')
death_df.head()

,UID,iso2,iso3,code3,FIPS,Admin2,Province_State,Country_Region,Lat,Long_,...,2/28/23,3/1/23,3/2/23,3/3/23,3/4/23,3/5/23,3/6/23,3/7/23,3/8/23,3/9/23
0,84001001,US,USA,840,1001.0,Autauga,Alabama,US,32.539527,-86.644082,...,230,232,232,232,232,232,232,232,232,232
1,84001003,US,USA,840,1003.0,Baldwin,Alabama,US,30.727750,-87.722071,...,724,726,726,726,726,726,726,726,727,727
2,84001005,US,USA,840,1005.0,Barbour,Alabama,US,31.868263,-85.387129,...,103,103,103,103,103,103,103,103,103,103
3,84001007,US,USA,840,1007.0,Bibb,Alabama,US,32.996421,-87.125115,...,109,109,109,109,109,109,109,109,109,109
4,84001009,US,USA,840,1009.0,Blount,Alabama,US,33.982109,-86.567906,...,261,261,261,261,261,261,261,261,261,261


In [5]:
case_df = case_df.drop(columns=['UID', 'iso2', 'iso3', 'code3', 'FIPS', 'Country_Region', 'Combined_Key'])
case_df = case_df.dropna(subset=['Admin2'])

case_df = pd.melt(case_df, 
                    id_vars=['Admin2', 'Province_State', 'Lat', 'Long_'],  # Keeping these as is
                    var_name='Date',  # New column to hold the date values
                    value_name='Cases')

case_df['Date'] = pd.to_datetime(case_df['Date'])

case_df.set_index('Date', inplace=True)

# Optimize memory usage
case_df['Cases'] = pd.to_numeric(case_df['Cases'], downcast='integer')
case_df['Lat'] = case_df['Lat'].apply(lambda x: Decimal(x).quantize(Decimal('0.001'), rounding=ROUND_HALF_UP))
case_df['Long_'] = case_df['Long_'].apply(lambda x: Decimal(x).quantize(Decimal('0.001'), rounding=ROUND_HALF_UP))

case_df['Admin2'] = case_df['Admin2'].astype('category')
case_df['Province_State'] = case_df['Province_State'].astype('category')


#Resample to weekly level
case_resample = case_df.groupby(['Admin2', 'Lat', 'Long_', 'Province_State']).resample('W').agg({
    'Cases': 'sum'      
}).reset_index()

#case_resample.sort_values(by='Date', inplace=True)
case_resample.reset_index(inplace=True, drop=True)

/var/folders/1k/mwcn6k811ks6zd4b6qh7g6nr0000gn/T/ipykernel_69001/180618039.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  case_df['Date'] = pd.to_datetime(case_df['Date'])


In [6]:
death_df = death_df.drop(columns=['UID', 'iso2', 'iso3', 'code3', 'FIPS', 'Country_Region', 'Combined_Key'])
death_df = death_df.dropna(subset=['Admin2'])

death_df = pd.melt(death_df, 
                    id_vars=['Admin2', 'Province_State', 'Lat', 'Long_', 'Population'],  # Keeping these as is
                    var_name='Date',  # New column to hold the date values
                    value_name='Deaths')

death_df['Date'] = pd.to_datetime(death_df['Date'])

death_df.set_index('Date', inplace=True)

# Optimize memory usage
death_df['Deaths'] = pd.to_numeric(death_df['Deaths'], downcast='integer')
death_df['Population'] = pd.to_numeric(death_df['Population'], downcast='integer')
death_df['Lat'] = death_df['Lat'].apply(lambda x: Decimal(x).quantize(Decimal('0.001'), rounding=ROUND_HALF_UP))
death_df['Long_'] = death_df['Long_'].apply(lambda x: Decimal(x).quantize(Decimal('0.001'), rounding=ROUND_HALF_UP))

death_df['Admin2'] = death_df['Admin2'].astype('category')
death_df['Province_State'] = death_df['Province_State'].astype('category')

#Resample to weekly level
death_resample = death_df.groupby(['Admin2', 'Lat', 'Long_', 'Province_State', 'Population']).resample('W').agg({
    'Deaths': 'sum'      
}).reset_index()

#death_resample.sort_values(by='Date', inplace=True)
death_resample.reset_index(inplace=True, drop=True)

/var/folders/1k/mwcn6k811ks6zd4b6qh7g6nr0000gn/T/ipykernel_69001/3758681767.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  death_df['Date'] = pd.to_datetime(death_df['Date'])


In [ ]:
death_staging = death_resample[['Date', 'Population', 'Deaths']]

base_df = pd.merge(case_resample, death_staging, on='Date', how='left')

base_df.head()